In [0]:
%python
# Força o Spark a usar o fuso horário de Brasília para datas e horas
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

In [0]:
%python
# dml/01_carga_dim_fundo_imobiliario.ipynb
# MAGIC %pip install requests pandas

# %%
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configurações de ambiente
catalogo = "product_dev"
schema = "financas"
tabela_dim = f"{catalogo}.{schema}.dim_fundo_imobiliario"

# Parametrização dinâmica do ano corrente para evitar quebra de pipeline em viradas de ano
ano_corrente = datetime.now().year

print(f"🚀 Iniciando Pipeline de Enriquecimento da Dimensão de FIIs ({ano_corrente})...")

# 2. Carrega a sua base atual de FIIs (B3)
print("2. Carregando dados cadastrais atuais de FIIs da tabela Dim...")
df_dim_atual = spark.sql(f"SELECT ticker, razao_social, nome_fundo, codigo_fundo, classificacao FROM {tabela_dim}")

# %%
# 3. Faz o download do arquivo ZIP do ano corrente diretamente da CVM
url_zip = f"https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_{ano_corrente}.zip"
headers = {'User-Agent': 'Mozilla/5.0'}

print(f"3. Baixando arquivo consolidado de {ano_corrente} diretamente da CVM...")
try:
    response = requests.get(url_zip, headers=headers, timeout=30)
    response.raise_for_status()
    print("   ✅ Download concluído com sucesso!")
except requests.exceptions.HTTPError as e:
    # Fallback defensivo para o ano anterior caso o arquivo do ano atual ainda não tenha sido gerado pela CVM
    print(f"   ⚠️ Arquivo de {ano_corrente} não encontrado (HTTP {e.response.status_code}). Tentando ano anterior...")
    ano_corrente = ano_corrente - 1
    url_zip = f"https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_{ano_corrente}.zip"
    response = requests.get(url_zip, headers=headers, timeout=30)
    response.raise_for_status()
    print(f"   ✅ Download de fallback ({ano_corrente}) concluído!")

# %%
# 4. Descompacta e lê o arquivo cadastral de dentro do ZIP
print(f"4. Abrindo arquivo compactado e lendo o arquivo cadastral 'geral' de {ano_corrente}...")
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
arquivo_geral = f"inf_mensal_fii_geral_{ano_corrente}.csv"

with zip_file.open(arquivo_geral) as f:
    df_pd_cvm = pd.read_csv(f, sep=';', encoding='ISO-8859-1', on_bad_lines='skip')

# Converte para Spark DataFrame tipando as colunas como String
df_spark_cvm_raw = spark.createDataFrame(df_pd_cvm.astype(str))

# %%
# 5. Limpa os dados da CVM, deduz tickers e seleciona a foto cadastral mais recente (Data_Referencia)
print("5. Tratando dados da CVM, extraindo Segmento de Atuação e isolando a competência (Data_Referencia) mais recente...")

# Janela de partição para identificar o informe cadastral mais recente de cada fundo baseado na Data_Referencia real do layout
window_spec = Window.partitionBy("ticker_cvm").orderBy(F.col("Data_Referencia").desc())

df_cvm_mapeado = (
    df_spark_cvm_raw
    # Filtra apenas ISINs válidos de FIIs B3
    .filter(
        F.col("Codigo_ISIN").startswith("BR") & 
        (~F.col("Codigo_ISIN").startswith("BR0000")) &
        (F.col("Codigo_ISIN") != "nan")
    )
    # Extrai o Ticker do ISIN (Ex: BRFVPQCTF015 -> FVPQ11)
    .withColumn("ticker_cvm", F.concat(F.substring(F.col("Codigo_ISIN"), 3, 4), F.lit("11")))
    # Garante o formato estrito de ticker B3 de varejo (4 letras + 11)
    .filter(F.col("ticker_cvm").rlike(r"^[A-Z]{4}11$"))
    # Remove as máscaras de formatação do CNPJ
    .withColumn("cnpj_limpo", F.regexp_replace(F.col("CNPJ_Fundo_Classe"), r"[\./-]", ""))
    # Aplica numeração de linha por data de competência (Data_Referencia) decrescente
    # Isso garante que o registro de maior data receberá row_num = 1
    .withColumn("row_num", F.row_number().over(window_spec))
    # Filtra apenas o registro de row_num 1 (garante foto cadastral mais recente e sem duplicados)
    .filter(F.col("row_num") == 1)
    # Seleciona e limpa as colunas necessárias para o de-para cadastral
    .select(
        F.col("ticker_cvm").alias("ticker_cvm"),
        F.col("cnpj_limpo").alias("cnpj"),
        F.col("Codigo_ISIN").alias("codigo_isin"),
        F.col("Nome_Fundo_Classe").alias("nome_fundo_cvm"),
        F.col("Nome_Administrador").alias("administrador"),
        # Captura o Segmento de Atuação oficial CVM com tratamento de nulos/vazios
        F.when(
            (F.col("Segmento_Atuacao") == "nan") | (F.col("Segmento_Atuacao").isNull()) | (F.trim(F.col("Segmento_Atuacao")) == ""), 
            F.lit("Não Informado")
        ).otherwise(F.col("Segmento_Atuacao")).alias("segmento_atuacao_cvm")
    )
)

# %%
# 6. Executa o LEFT JOIN para enriquecer a sua tabela original (Garante zero perda de dados da B3)
print("6. Cruzando bases de dados (B3 + CVM) via LEFT JOIN...")
df_dim_enriquecida = (
    df_dim_atual.alias("b3")
    .join(
        df_cvm_mapeado.alias("cvm"),
        F.col("b3.ticker") == F.col("cvm.ticker_cvm"),
        "left"
    )
    .select(
        F.col("b3.ticker").alias("ticker"),
        F.col("b3.razao_social").alias("razao_social"),
        F.col("b3.nome_fundo").alias("nome_fundo"),
        F.col("b3.codigo_fundo").alias("codigo_fundo"),
        F.col("b3.classificacao").alias("classificacao"),
        # Campos enriquecidos da CVM
        F.col("cvm.cnpj").alias("cnpj"),
        F.col("cvm.codigo_isin").alias("codigo_isin"),
        F.col("cvm.nome_fundo_cvm").alias("nome_fundo_cvm"),
        F.col("cvm.administrador").alias("administrador"),
        # Campo de Segmento de Atuação da CVM (com fallback para a Classificação B3 original do Fundo caso o join seja nulo)
        F.coalesce(
            F.col("cvm.segmento_atuacao_cvm"), 
            F.col("b3.classificacao")
        ).alias("segmento_atuacao"),
        # Metadado de auditoria
        F.current_timestamp().alias("data_carga")
    )
)

# %%
# 7. Gravação física definitiva na tabela Delta original usando INSERT OVERWRITE
print(f"7. Gravando dados enriquecidos de forma atômica na tabela Delta '{tabela_dim}'...")

# Escreve os dados fazendo o overwrite de forma atômica na tabela Delta
df_dim_enriquecida.write.format("delta").mode("overwrite").saveAsTable(tabela_dim)

print("✅ PIPELINE CONCLUÍDO COM SUCESSO!")
print("   Sua dimensão de FIIs está física, definitiva e temporalmente enriquecida com o Segmento de Atuação REAL da CVM!")

# %%
# 8. Amostra para validação visual rápida do schema e conteúdo
display(
    spark.sql(f"""
        SELECT ticker, nome_fundo, classificacao, segmento_atuacao, cnpj, data_carga 
        FROM {tabela_dim} 
        WHERE cnpj IS NOT NULL 
        LIMIT 10
    """)
)

In [0]:
print(df_pd_cvm.columns.tolist())

In [0]:
%sql
select * from product_dev.financas.stg_scoring_tijolo tablesample (5 rows)

In [0]:
%sql
select segmento_atuacao from product_dev.financas.dim_fundo_imobiliario
group by segmento_atuacao

In [0]:
# dml/01_carga_dim_fundo_imobiliario.ipynb
# MAGIC %pip install requests pandas

# %%
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configurações de ambiente
catalogo = "product_dev"
schema = "financas"
tabela_dim = f"{catalogo}.{schema}.dim_fundo_imobiliario"

# Parametrização dinâmica do ano corrente para evitar quebra de pipeline em viradas de ano
ano_corrente = datetime.now().year

print(f"🚀 Iniciando Pipeline de Enriquecimento da Dimensão de FIIs ({ano_corrente})...")

# 2. Carrega a sua base atual de FIIs (B3)
print("2. Carregando dados cadastrais atuais de FIIs da tabela Dim...")
df_dim_atual = spark.sql(f"SELECT ticker, razao_social, nome_fundo, codigo_fundo, classificacao FROM {tabela_dim}")

# %%
# 3. Faz o download do arquivo ZIP do ano corrente diretamente da CVM
url_zip = f"https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_{ano_corrente}.zip"
headers = {'User-Agent': 'Mozilla/5.0'}

print(f"3. Baixando arquivo consolidado de {ano_corrente} diretamente da CVM...")
try:
    response = requests.get(url_zip, headers=headers, timeout=30)
    response.raise_for_status()
    print("   ✅ Download concluído com sucesso!")
except requests.exceptions.HTTPError as e:
    # Fallback defensivo para o ano anterior caso o arquivo do ano atual ainda não tenha sido gerado pela CVM (ex: início de janeiro)
    print(f"   ⚠️ Arquivo de {ano_corrente} não encontrado (HTTP {e.response.status_code}). Tentando ano anterior...")
    ano_corrente = ano_corrente - 1
    url_zip = f"https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_{ano_corrente}.zip"
    response = requests.get(url_zip, headers=headers, timeout=30)
    response.raise_for_status()
    print(f"   ✅ Download de fallback ({ano_corrente}) concluído!")

# %%
# 4. Descompacta e lê o arquivo cadastral de dentro do ZIP
print(f"4. Abrindo arquivo compactado e lendo o arquivo cadastral 'geral' de {ano_corrente}...")
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
arquivo_geral = f"inf_mensal_fii_geral_{ano_corrente}.csv"

with zip_file.open(arquivo_geral) as f:
    df_pd_cvm = pd.read_csv(f, sep=';', encoding='ISO-8859-1', on_bad_lines='skip')

# Converte para Spark DataFrame tipando as colunas como String
df_spark_cvm_raw = spark.createDataFrame(df_pd_cvm.astype(str))

# %%
# 5. Limpa os dados da CVM, deduz tickers e seleciona a foto cadastral mais recente (DT_COMPT)
print("5. Tratando dados da CVM, extraindo Segmento de Atuação e isolando a competência (DT_COMPT) mais recente...")

# Janela de partição para identificar o informe mensal cadastral mais recente de cada fundo
window_spec = Window.partitionBy("ticker_cvm").orderBy(F.col("DT_COMPT").desc())

df_cvm_mapeado = (
    df_spark_cvm_raw
    # Filtra apenas ISINs válidos de FIIs B3
    .filter(
        F.col("Codigo_ISIN").startswith("BR") & 
        (~F.col("Codigo_ISIN").startswith("BR0000")) &
        (F.col("Codigo_ISIN") != "nan")
    )
    # Extrai o Ticker do ISIN (Ex: BRFVPQCTF015 -> FVPQ11)
    .withColumn("ticker_cvm", F.concat(F.substring(F.col("Codigo_ISIN"), 3, 4), F.lit("11")))
    # Garante o formato estrito de ticker B3 de varejo (4 letras + 11)
    .filter(F.col("ticker_cvm").rlike(r"^[A-Z]{4}11$"))
    # Remove as máscaras de formatação do CNPJ
    .withColumn("cnpj_limpo", F.regexp_replace(F.col("CNPJ_Fundo_Classe"), r"[\./-]", ""))
    # Aplica numeração de linha por data de competência (DT_COMPT) decrescente
    # Isso garante que o registro de maior data receberá row_num = 1
    .withColumn("row_num", F.row_number().over(window_spec))
    # Filtra apenas o registro de row_num 1 (garante foto cadastral mais recente e sem duplicados)
    .filter(F.col("row_num") == 1)
    # Seleciona e limpa as colunas necessárias para o de-para cadastral
    .select(
        F.col("ticker_cvm").alias("ticker_cvm"),
        F.col("cnpj_limpo").alias("cnpj"),
        F.col("Codigo_ISIN").alias("codigo_isin"),
        F.col("Nome_Fundo_Classe").alias("nome_fundo_cvm"),
        F.col("Nome_Administrador").alias("administrador"),
        # Captura o Segmento de Atuação e trata fallbacks para nulos ou Strings vazias
        F.when(
            (F.col("Segmento_Atuacao") == "nan") | (F.col("Segmento_Atuacao").isNull()) | (F.trim(F.col("Segmento_Atuacao")) == ""), 
            F.lit("Não Informado")
        ).otherwise(F.col("Segmento_Atuacao")).alias("segmento_atuacao")
    )
)

# %%
# 6. Executa o LEFT JOIN para enriquecer a sua tabela original (Garante zero perda de dados da B3)
print("6. Cruzando bases de dados (B3 + CVM) via LEFT JOIN...")
df_dim_enriquecida = (
    df_dim_atual.alias("b3")
    .join(
        df_cvm_mapeado.alias("cvm"),
        F.col("b3.ticker") == F.col("cvm.ticker_cvm"),
        "left"
    )
    .select(
        F.col("b3.ticker").alias("ticker"),
        F.col("b3.razao_social").alias("razao_social"),
        F.col("b3.nome_fundo").alias("nome_fundo"),
        F.col("b3.codigo_fundo").alias("codigo_fundo"),
        F.col("b3.classificacao").alias("classificacao"),
        # Campos enriquecidos da CVM
        F.col("cvm.cnpj").alias("cnpj"),
        F.col("cvm.codigo_isin").alias("codigo_isin"),
        F.col("cvm.nome_fundo_cvm").alias("nome_fundo_cvm"),
        F.col("cvm.administrador").alias("administrador"),
        # Campo de Segmento de Atuação da CVM (com tratamento de fallback)
        F.coalesce(F.col("cvm.segmento_atuacao"), F.lit("Não Informado")).alias("segmento_atuacao"),
        # Metadado de auditoria
        F.current_timestamp().alias("data_carga")
    )
)

# %%
# 7. Gravação física definitiva na tabela Delta original usando INSERT OVERWRITE
print(f"7. Gravando dados enriquecidos de forma atômica na tabela Delta '{tabela_dim}'...")

# Escreve os dados fazendo o overwrite de forma atômica na tabela Delta
df_dim_enriquecida.write.format("delta").mode("overwrite").saveAsTable(tabela_dim)

print("✅ PIPELINE CONCLUÍDO COM SUCESSO!")
print("   Sua dimensão de FIIs está física, definitiva e temporalmente enriquecida com dados da CVM!")

# %%
# 8. Amostra para validação visual rápida do schema e conteúdo
display(
    spark.sql(f"""
        SELECT ticker, nome_fundo, classificacao, segmento_atuacao, cnpj, data_carga 
        FROM {tabela_dim} 
        WHERE cnpj IS NOT NULL 
        LIMIT 10
    """)
)